[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeteoSwiss/opendata-nwp-demos/blob/main/09_pollen_conversion.ipynb)

# Retrieving, Converting, and Visualizing ICON-CH2-EPS Pollen Forecast Data

This notebook demonstrates the full workflow for accessing, converting, and visualizing determinstic ICON-CH2-EPS pollen forecast data. The data is provided by MeteoSwiss as part of Switzerland’s [Open Government Data (OGD) initiative](https://www.meteoswiss.admin.ch/services-and-publications/service/open-data.html).

Pollen data is provided as specific number concentration — the number of particles per kg of air. But in practice, almost always the volume concentration (number of particles per volume) is desired. In this notebook this conversion is demonstrated, either by using model data or with a significantly cheaper approximation.

For visualization, this notebook uses the [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/examples/guide/01-introduction.html) library developed by ECMWF, which offers intuitive plotting tools for meteorological data.

---

## 🔍 **What This Notebook Covers:**

 📥  **Retrieve**  
    Fetch deterministic ICON-CH2-EPS forecast data (pollen and variables required for conversion to volume concentration) via the `ogd_api` module.

 📐  **Convert to volume concentration**  
    Convert specific pollen number concentration to volume concentration using model data with [meteodata-lab](https://meteoswiss.github.io/meteodata-lab/) operators, or with a simple approximation.
 

 🧭  **Regrid**  
    Interpolate ICON-CH2-EPS data from its native, icosahedral grid to the regular latitude/longitude grid [WGS84 (EPSG:4326)](https://epsg.io/4326).

 🌍  **Visualize**  
    Plot the processed data on a map using [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/examples/guide/01-introduction.html).

---

## Retrieving Pollen Forecast
In this first part, we retrieve deterministic ICON-CH2-EPS pollen forecast data. To access this data, we use the `ogd_api` module from the [meteodata-lab](https://meteoswiss.github.io/meteodata-lab/) library — a convenient interface for accessing numerical weather forecasts via the [STAC (SpatioTemporal Asset Catalog) API](https://data.geo.admin.ch/api/stac/static/spec/v1/apitransactional.html#tag/Data/operation/getAsset), which provides structured access to Switzerland’s open geospatial data.

## 📦 Install Dependencies (Colab only)
This cell installs all required dependencies if you're running the notebook in Google Colab. It is automatically skipped when running in a local Jupyter environment.

In [ ]:
# 📦 Google Colab Setup (skipped if not running in Colab)
import sys

def is_colab():
    return "google.colab" in sys.modules

if is_colab():
    !git clone https://github.com/MeteoSwiss/opendata-nwp-demos.git
    %cd opendata-nwp-demos
    !pip install poetry && poetry config virtualenvs.in-project true && poetry install --no-ansi
    import sys, os, pathlib
    venv = pathlib.Path(".venv")
    site = venv / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
    sys.path.insert(0, str(site))
    os.environ["ECCODES_DEFINITION_PATH"] = str((venv / "share/eccodes-cosmo-resources/definitions").resolve())

## 📥 Retrieve ICON-CH2-EPS Pollen Data

We request the pollen variable `TODO`, in addition to temperature `T`, pressure `P`, specific humidity `QV` and the cloud mixing ratio `QC`. Those additional variables will later be used to compute the total air density which is needed for conversion to the volume number density.

>⏰ **Forecast Availability**: Forecast data will typically be available a couple of hours after the reference time — due to the model runtime and subsequent upload time. The data remains accessible for 24 hours after upload.

>⏰ **Pollen Data Availability**: Pollen data is only available during the Pollen season from <TODO: date> through <TODO: date>.

In [ ]:
from meteodatalab import ogd_api

# TODO: replace TOT_PREC with pollen var
param_list = ['TOT_PREC', 'T', 'P', 'QV', 'QC']
reqlist = []

for param in param_list:
    req = ogd_api.Request(
        collection="ogd-forecasting-icon-ch2",
        variable=param,
        ref_time="latest",
        perturbed=False,
        lead_time="P0DT6H",
    )
    reqlist.append((param, req))

> 💡 **Tip**: Use temporary caching with earthkit-data to skip repeated downloads — it's auto-cleaned after the session.
> *For more details, see the [earthkit-data caching docs](https://earthkit-data.readthedocs.io/en/latest/examples/cache.html)*.

> 💡 **Hint**: If you get an error message containing `HTTPError: 403 Client Error: Forbidden for url`, you may be trying to retrieve data older than 24h hours! Please adjust your requests.

In [ ]:
from earthkit.data import config
config.set("cache-policy", "temporary")

da_dict = {}
for param, req in reqlist:
    da = ogd_api.get_from_ogd(req)
    da_dict[param] = da

## 📐 Convert to volume concentration

The volume number concentration is obtained by multiplying the specific number concentration by the total air density $\rho$. However, the density itself must be computed first.

### **Approach 1**: Compute Total Air Density from the Model

On the grid, the pollen variables are associated with the lowest (full) level. From the 3d variables, only this level is selected to proceed.

> ⚠️ **Surface variables**: The pollen variables are *not* not located on the surface. For temperature and pressure, 2d (surface) variables exist, but are different from the lowest level's values. TODO: add some more rationale about how suitable the 2d vars PS, T_G, or T_2M would be as alternative to the 3d fields T and P.

In [ ]:
from meteodatalab.operators import rho as rho

# compute total air density in the lowest level only
density_model = rho.compute_rho_tot(da_dict['T'].isel(z=-1),
                                    da_dict['P'].isel(z=-1),
                                    da_dict['QV'].isel(z=-1),
                                    da_dict['QC'].isel(z=-1))


### **Approach 2**: Approximate Air Density as a Function of Height

Instead of using model data to compute the air density, we can also just crudely approximate it as a function of height. This way, we can avoid downloading several 3d fields only to determine the density in the lowest level.
The math behind this is outlined, e.g., in [Wikipedia](https://en.wikipedia.org/wiki/Density_of_air#Troposphere).

In [ ]:
import xarray as xr
import meteodatalab.metadata as metadata

# define the approximation
def approx_z2rho(z: xr.DataArray):
    p0 = 101325   # sea level standard atmospheric pressure
    M = 0.0289652 # molar mass of dry air
    R = 8.31446   # ideal gas constant
    T0 = 288.15   # sea level standard temperature
    L = 0.0065    # temperature lapse rate
    g = 9.80665   # earth surface gravitational acceleration
    return xr.DataArray(
        data=p0*M/(R*T0) * (1 - L*z/T0)**((g*M/(R*L) - 1)),
        attrs=metadata.override(z.metadata, shortName="DEN")
    )

In [ ]:
from meteodatalab import ogd_api
from meteodatalab import grib_decoder, data_source
import xarray as xr

# fetch the height of the surface
url = ogd_api.get_collection_asset_url(
    collection_id="ch.meteoschweiz.ogd-forecasting-icon-ch2",
    asset_id="horizontal_constants_icon-ch2-eps.grib2"
)

ds2 = grib_decoder.load(
    source=data_source.URLDataSource(urls=[url]),
    request={"param": "HSURF"},
    geo_coords=lambda uuid: {}
)

# the lowest level is approximately 20m high, so the pollen's position is 10m above the ground
with xr.set_options(keep_attrs=True):
    z_pollen = ds2['HSURF'] + 10

In [ ]:
# call the density approximation
density_approx = approx_z2rho(z_pollen)

# remove the lead_time dimension, since this approximation is constant
density_approx = density_approx.squeeze(dim='lead_time', drop=True)

# for later regridding, assign the lon and lat coordinates
density_approx = density_approx.assign_coords(lon=da_dict['TOT_PREC']["lon"], lat=da_dict['TOT_PREC']["lat"])

### Convert to volume number concentration

In [ ]:
# conversion to volume number concentration: multiply with density
with xr.set_options(keep_attrs=True):
    pollen_vol_model = da_dict['TOT_PREC'] * density_model # TODO: replace 'TOT_PREC', fix units
    pollen_vol_approx = da_dict['TOT_PREC'] * density_approx # TODO: replace 'TOT_PREC', fix units

## 🗺️ Visualize the data

In [ ]:
# Define regridding
from rasterio.crs import CRS
from meteodatalab.operators import regrid

# Define ~1 km target grid over ICON-CH1-EPS domain
extent = (-0.817, 18.183, 41.183, 51.183)  # (xmin, xmax, ymin, ymax)
nx, ny = 732, 557

# Create regular lat/lon grid (EPSG:4326)
destination = regrid.RegularGrid(CRS.from_epsg(4326), nx, ny, *extent)

### Error of the density approximation

In [ ]:
# compute relative error
with xr.set_options(keep_attrs=True):
    density_error = density_approx/density_model

# regrid
density_error_regrid = regrid.iconremap(density_error, destination)

In [ ]:
import pandas as pd
import cartopy.crs as ccrs
from earthkit.plots.geo import bounds, domains
from earthkit.plots.styles import Style
import earthkit
import matplotlib.colors as mcolors

# === Setup Domain (Switzerland) ===
bbox = bounds.BoundingBox(5.7, 10.5, 45.6, 48, ccrs.Geodetic())
domain = domains.Domain.from_bbox(bbox=bbox)

# === Create Map ===
chart = earthkit.plots.Map(domain=domain)

contourf_style = Style(
    norm=mcolors.CenteredNorm(
        vcenter=1.0
    ),
    colors="RdBu_r",
    legend_style="colorbar",
)
chart.contourf(density_error_regrid, x="lon", y="lat", style=contourf_style)


# === Add Map Features ===
chart.land()
chart.coastlines()
chart.borders()
chart.gridlines()


# === Annotate Chart ===
ref_time = pd.to_datetime(density_error_regrid.coords["ref_time"].values[0]).strftime("%Y-%m-%d %H:%M UTC")
lead_time = density_error_regrid.coords["lead_time"].values.astype('timedelta64[h]')

title = f"Relative error of density approximation | {ref_time} (+{lead_time})"
legend_label = f"Relative error (approx/model)"

chart.title(title)
chart.legend(label=legend_label)


# === Show Plot ===
chart.show()


### Visualize Pollen (TODO)

In [ ]:
# regrid
da_geo = regrid.iconremap(pollen_vol_model, destination)

In [ ]:
import pandas as pd
import cartopy.crs as ccrs
from earthkit.plots.geo import bounds, domains
import earthkit
from earthkit.plots.styles import Style
import numpy as np


# === Setup Domain (Switzerland) ===
bbox = bounds.BoundingBox(5.7, 10.5, 45.6, 48, ccrs.Geodetic())
domain = domains.Domain.from_bbox(bbox=bbox)


# === Create Map ===
chart = earthkit.plots.Map(domain=domain)

#levels = np.linspace(0.8, 1.3, 11)

contourf_style = Style(
    #levels=levels,
    colors="YlGnBu",
    legend_style="colorbar",
)
chart.contourf(da_geo, x="lon", y="lat", style=contourf_style)


# === Add Map Features ===
chart.land()
chart.coastlines()
chart.borders()
chart.gridlines()


# === Annotate Chart ===
ref_time = pd.to_datetime(da_geo.coords["ref_time"].values[0]).strftime("%Y-%m-%d %H:%M UTC")
lead_time = da_geo.coords["lead_time"].values.astype('timedelta64[h]')

parameter = da_geo.attrs["parameter"]
title = f"TODO Title | {ref_time} (+{lead_time})"
legend_label = f"{parameter['name']} ({'TODO unit'})"

chart.title(title)
chart.legend(label=legend_label)


# === Show Plot ===
chart.show()
